## function which drops modalities and has the option of an invariant slot 

In [2]:
import random
import torch 
from collections import Counter
import numpy as np 



def rand_set_channels_to_zero(dataset_modalities: list, batch_img_data: torch.Tensor, domain_invariant:bool) -> tuple[list[int], torch.Tensor]:
    """Randomly set a subset of channels to zero
    return a list of modalities remaining(after drop) and an image tensors from remaining channels """
    modalities_remaining=[]
    
    if domain_invariant:
        
        # append a new channel to dimension 1
        batch_img_da = torch.cat((batch_img_data, torch.zeros((batch_img_data.shape[0], 1, 128, 128, 128))), dim=1)
        dataset_modalities.append(len(dataset_modalities))
        print(f'batch size after additional slot added: {batch_img_da.size()}')
        print(dataset_modalities)
    else:
        batch_img_da = batch_img_data

    
   
    for i in range (batch_img_da.shape[0]):   

        if domain_invariant:
            # start from 1 dropped as added modality is zeros so is effectively dropped. 
            number_of_dropped_modalities = np.random.randint(1,len(dataset_modalities))
        else:
            number_of_dropped_modalities = np.random.randint(0,len(dataset_modalities))    
        
        modalities_dropped = random.sample(list(np.arange(len(dataset_modalities))), number_of_dropped_modalities)    
        modalities_dropped.sort()
        print(f'modalities_dropped: {modalities_dropped}')
       
       # odds of getting a zero is 1/len(dataset_modalities)
        

        # copy of batch image data
        batch_img = batch_img_da.clone()
        print(f'batch_img: {batch_img.shape}')

        print(f'batch_img_data: {batch_img_da.shape}')
       
       
        # multiplied dropped channels by zero. 
        batch_img_da[i,modalities_dropped,:,:,:] = 0.

        print(f'BATCHHHHHH: {batch_img_da.size()}')
        
        modalities_remaining.append(list(set(np.arange(len(dataset_modalities))) - set(modalities_dropped)))   

        print(f'modalities_remaining: {modalities_remaining}')

        if domain_invariant:
            
            if len(modalities_dropped) == 0:
                continue
            #TODO: look at this and probability distribution. 

            elif len (modalities_dropped) > 0:

               
                channel_add = random.sample(modalities_dropped, 1)
                
                print(f'channel_add: {channel_add}')
                
                invar = batch_img[i,channel_add,:,:,:]  
                invar = torch.unsqueeze(invar,1)

                print(f'invar: {invar.size()}')       
                print(batch_img.size())       
                print(f'invariant_slot: {batch_img.size()}')       
                print(f'batch_img_data: {batch_img_da.shape}')
                # create extra channel dimension 
                #iny = batch_img[i][channel_add]   #[i,channel,:,:,:]  
                batch_img_da[i,[len(dataset_modalities)-1],:,:,:]=invar
                print(batch_img_da[i][0])
                print(batch_img_da[i][1])
                print(batch_img_da[i][2])
                print(batch_img_da[i][3])
                print(batch_img_da[i][4])
                
                torch.allclose(batch_img_da[i][len(dataset_modalities)-1],batch_img[i][channel_add])
                    
                

    return modalities_remaining, batch_img_da

      
       



#print(f' before additonal slot added: { input_data.shape}')

domain_invariant = True

total_modalities_present = []

for z in range(1000):
    input_data = torch.rand(2, 4, 128, 128, 128)
    dataset_list= [0, 1, 2, 3]
    modalities_remaining, batch_img_datum =rand_set_channels_to_zero(dataset_list, input_data,domain_invariant)
    # plot bar chart for number of times that each modality is pressent
    # create dictionary for each modality and number of times that it is present
    # plot the dictionary
    flat_list = [item for sublist in modalities_remaining for item in sublist]
    for i in range(len(modalities_remaining)):
        for j in modalities_remaining[i]:
            total_modalities_present.append(j)
    
print(f'modalities_remaining: {total_modalities_present}')
print(f' Count of modalities remaining: {Counter(total_modalities_present)}')
# correct shape
print(batch_img_datum.shape)





batch size after additional slot added: torch.Size([2, 5, 128, 128, 128])
[0, 1, 2, 3, 4]
modalities_dropped: [0, 1, 4]
batch_img: torch.Size([2, 5, 128, 128, 128])
batch_img_data: torch.Size([2, 5, 128, 128, 128])
BATCHHHHHH: torch.Size([2, 5, 128, 128, 128])
modalities_remaining: [[2, 3]]
channel_add: [1]
invar: torch.Size([1, 1, 128, 128, 128])
torch.Size([2, 5, 128, 128, 128])
invariant_slot: torch.Size([2, 5, 128, 128, 128])
batch_img_data: torch.Size([2, 5, 128, 128, 128])
tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        [[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0.,

batch size after additional slot added: torch.Size([2, 5, 128, 128, 128])
[0, 1, 2, 3, 4]
modalities_dropped: [0, 1, 2, 3]
batch_img: torch.Size([2, 5, 128, 128, 128])
batch_img_data: torch.Size([2, 5, 128, 128, 128])
BATCHHHHHH: torch.Size([2, 5, 128, 128, 128])
modalities_remaining: [[4]]
channel_add: [0]
invar: torch.Size([1, 1, 128, 128, 128])
torch.Size([2, 5, 128, 128, 128])
invariant_slot: torch.Size([2, 5, 128, 128, 128])
batch_img_data: torch.Size([2, 5, 128, 128, 128])
tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        [[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0.,

KeyboardInterrupt: 

## Stream line version of function which drops modalities and has the option of an invariant slot 



In [8]:
import random
import torch 
from collections import Counter
import numpy as np 

# streamline version

def rand_set_channels_to_zero(dataset_modalities: list, batch_img_data: torch.Tensor, domain_invariant:bool) -> tuple[list[int], torch.Tensor]:
    """Randomly set a subset of channels to zero
    return a list of modalities remaining(after drop) and an image tensors from remaining channels """
    modalities_remaining=[]
    
    if domain_invariant:
        # append a new channel to dimension 1
        batch_img_da = torch.cat((batch_img_data, torch.zeros((batch_img_data.shape[0], 1, 128, 128, 128))), dim=1)
        dataset_modalities.append(len(dataset_modalities))
        print(f'Batch size after additional slot added: {batch_img_da.size()}')
    else:
        batch_img_da = batch_img_data

    for i in range (batch_img_da.shape[0]):   

        if domain_invariant:
            # start from 1 dropped as added modality is zeros so is effectively dropped. 
            number_of_dropped_modalities = np.random.randint(1,len(dataset_modalities))
        else:
            number_of_dropped_modalities = np.random.randint(0,len(dataset_modalities))    
        
        modalities_dropped = random.sample(list(np.arange(len(dataset_modalities))), number_of_dropped_modalities)    
        modalities_dropped.sort()
        print(f'modalities_dropped: {modalities_dropped}')
        # copy of batch image data
        batch_img = batch_img_da.clone()

        # multiplied dropped channels by zero. 
        batch_img_da[i,modalities_dropped,:,:,:] = 0.
        modalities_remaining.append(list(set(np.arange(len(dataset_modalities))) - set(modalities_dropped)))   
        print(f'modalities_remaining: {modalities_remaining}')

        if domain_invariant:
            
            channel_add = random.sample(modalities_dropped, 1)
            
            print(f'channel_add: {channel_add}')
            invar = batch_img[i,channel_add,:,:,:]  
            print(f' channel being added {batch_img[i,channel_add,:,:,:]} ' )
            invar = torch.unsqueeze(invar,1)
            batch_img_da[i,[len(dataset_modalities)-1],:,:,:]=invar
            batch_img_da[i,[-1],:,:,:]=invar      
            # double check that the invariant slot is the same as the selected dropped channel
            result = torch.allclose(batch_img_da[i][-1],batch_img[i][channel_add])
            print(f"Assert selected dropped channel is coancatenated: {result}")
                
                

    return modalities_remaining, batch_img_da

      
##############################################################   test function  ####################################################################

#print(f' before additonal slot added: { input_data.shape}')

domain_invariant = True

total_modalities_present = []

for z in range(1):
    input_data = torch.rand(2, 4, 128, 128, 128)
    dataset_list= [0, 1, 2, 3]
    modalities_remaining, batch_img_datum =rand_set_channels_to_zero(dataset_list, input_data,domain_invariant)
  
    # plot bar chart for number of times that each modality is pressent
    # create dictionary for each modality and number of times that it is present
    # plot the dictionary
    flat_list = [item for sublist in modalities_remaining for item in sublist]
    for i in range(len(modalities_remaining)):
        for j in modalities_remaining[i]:
            total_modalities_present.append(j)
    
print(f'modalities_remaining: {total_modalities_present}')
print(f' Count of modalities remaining: {Counter(total_modalities_present)}')
# correct shape
print(batch_img_datum.shape)

Batch size after additional slot added: torch.Size([2, 5, 128, 128, 128])
modalities_dropped: [0, 1, 2, 3]
modalities_remaining: [[4]]
channel_add: [3]
 channel being added tensor([[[[8.1517e-01, 4.3425e-01, 5.7931e-01,  ..., 2.4646e-01,
           6.1464e-01, 9.9651e-01],
          [7.8089e-01, 8.0340e-01, 1.8272e-01,  ..., 5.8864e-01,
           2.4456e-01, 4.6379e-01],
          [4.0415e-01, 2.0458e-01, 5.6429e-01,  ..., 6.6655e-01,
           3.9124e-01, 5.0943e-01],
          ...,
          [5.4278e-01, 2.0947e-01, 5.5119e-02,  ..., 3.0767e-01,
           5.9415e-01, 6.2718e-02],
          [8.7948e-01, 6.8287e-01, 2.8614e-01,  ..., 8.4009e-01,
           2.8248e-01, 4.1879e-01],
          [6.5936e-01, 3.1276e-01, 2.6190e-01,  ..., 5.6179e-02,
           7.5070e-01, 5.1221e-01]],

         [[3.6398e-01, 6.6784e-01, 1.4264e-01,  ..., 9.1591e-01,
           7.6721e-01, 5.1474e-01],
          [9.4418e-01, 6.1452e-01, 4.3204e-01,  ..., 2.4731e-01,
           2.1419e-01, 2.1962e-01],
  

## function/ investigation into one slot for all modalities 

In [1]:
# do not need drop for this.
# TODO: iterate through all the modalities and select and place into slot. (do I want to zero out the rest of slots or literally have one slot)

# set no drop.

# randomly select a channel from all modalities present and then randomly select from a dataset to extract this data from and place into channel 
# and then train with one slot and blank out all the rest usig slicing. 

#TODO: whilst other models up and running.  

# literally have one slot and iterate though all modalities and place into the slot. 

# total slot = 1 , size of input to unet will be one channel.
# set drop to zero and iterate all modalities and place into slot
# randomly shuffle- yes. 

# train with one slot = true.





In [7]:
# test single slot function. 


def single_slot( batch_img_data: torch.Tensor) -> tuple[list[int], torch.Tensor]:
    """ randomly select one channel and place into single invariant slot and then retutn the image tebsir wuith one channel"""

    batch_size, num_channels, height, width, depth = batch_img_data.shape

    # Initialize the output tensor of 1 channel with zeros
    batch_img_da = torch.zeros((batch_size, 1, height, width, depth), dtype=batch_img_data.dtype, device=batch_img_data.device)

    for i in range(batch_size):
        # Randomly select one channel to keep
        selected_channel = random.randint(0, num_channels - 1)

        # print all channels and confirm correct one is being left. 
        print(f'selected_channel: {selected_channel}')

        print(batch_img_data[i, selected_channel, :, :, :])

        # Copy the selected channel to the output tensor
        batch_img_da[i, 0, :, :, :] = batch_img_data[i, selected_channel, :, :, :]

        print(f'batch_img_data: {batch_img_da.size()}')
        print(batch_img_da[i,0,:,:,:])
    print(f'batch_img_data: {batch_img_da.size()}')         

    return batch_img_da

#test funciton 

input_data = torch.rand(2, 10, 128, 128, 128)
output_data = single_slot(input_data)


selected_channel: 3
tensor([[[2.2173e-01, 1.0885e-01, 8.5691e-01,  ..., 1.5241e-01,
          1.2176e-01, 7.7698e-01],
         [6.5208e-01, 3.0668e-01, 2.5071e-01,  ..., 8.3132e-01,
          1.5503e-01, 9.1435e-03],
         [6.7969e-02, 7.0424e-01, 8.7133e-01,  ..., 6.7356e-01,
          1.8245e-01, 5.2427e-01],
         ...,
         [9.8463e-02, 5.5016e-01, 3.5048e-01,  ..., 2.1411e-01,
          8.7972e-01, 4.8373e-01],
         [4.4749e-01, 2.5884e-01, 7.4737e-01,  ..., 7.5553e-01,
          4.6093e-01, 5.5755e-01],
         [9.2364e-01, 9.0237e-01, 3.7238e-01,  ..., 6.6792e-01,
          5.9351e-01, 5.6166e-01]],

        [[1.0899e-01, 1.3558e-01, 1.6897e-01,  ..., 3.3474e-01,
          7.6848e-01, 5.5027e-01],
         [3.9271e-01, 1.4410e-01, 2.1260e-02,  ..., 6.7479e-01,
          5.6226e-01, 3.3787e-01],
         [4.7394e-01, 3.0626e-01, 8.6179e-01,  ..., 4.2215e-01,
          1.7116e-01, 3.1612e-01],
         ...,
         [4.2470e-01, 1.0548e-01, 8.8348e-02,  ..., 6.2164e

# Adversarial loss


In [ ]:
# calculate adversail loss for each modality to generate more data for underrepresented modalities. 
# for example calulate say use 5 datsets and have 6 different modalities. for ATLAS it only has T1 modality is it possible 
# to generate the missing modalities. and use a disciminator for style form other modalities.
# potentially use a cyclegan to generate the misisng modalitiy.
# want to keep the structure of current modality but change the style to modality in question. 
# can apply style transfer to only area of interest i.e can use the mask of the brain to maintain the overall structure. Constrained Style Transfer. 
# 




### Addinng an additioal slot for finetuning unseen modality
### Randomly initialize the weigths


In [2]:
import torch

# load pretrained model and save weights and add a new channel to the first layer of the model.

# Load the pre-trained model
pre_trained_model = (
    "models/all_in_one/TBI/standard_random_drop_1_2024-12-17_11-07_Epoch_599.pth"
)
model = torch.load(pre_trained_model)
print(model)

# Get the input size of the first convolutional layer
input_size = model["conv_1.conv.unit0.conv.weight"].shape[1]
print(f"Input channels of model: {input_size}")

# Add a new input channel to the first layer of the model
num_input_channels = input_size + 1

# Create a new Conv3d layer with an additional input channel
new_conv = torch.nn.Conv3d(
    num_input_channels, 16, kernel_size=3, stride=1, padding=1, bias=True
)

# Copy the weights from the old conv layer to the new conv layer
with torch.no_grad():
    new_conv.weight[:, :input_size, :, :, :] = model["conv_1.conv.unit0.conv.weight"]
    new_conv.bias = torch.nn.Parameter(
        model["conv_1.conv.unit0.conv.bias"]
    )  # Convert to torch.nn.Parameter

# Replace the old conv layer with the new conv layer in the model
model["conv_1.conv.unit0.conv.weight"] = new_conv.weight
model["conv_1.conv.unit0.conv.bias"] = new_conv.bias

print(model["conv_1.conv.unit0.conv.weight"].shape)

OrderedDict([('conv_1.conv.unit0.conv.weight', tensor([[[[[-1.8650e-02, -2.9946e-02, -1.4751e-01],
           [ 5.2785e-02, -7.0971e-02, -2.0682e-01],
           [ 3.8224e-02, -3.8411e-03, -1.4006e-01]],

          [[-1.2245e-02, -7.6076e-02, -2.1979e-01],
           [-1.3038e-02, -2.8850e-01, -2.2201e-01],
           [-2.5550e-02, -8.4258e-02, -2.2440e-01]],

          [[-1.9022e-04, -1.7043e-01, -1.5890e-01],
           [ 2.6375e-03, -1.0390e-01, -1.9219e-01],
           [-6.8812e-03, -1.4187e-01, -1.8321e-01]]],


         [[[ 9.9163e-02, -8.6958e-02, -1.3631e-01],
           [ 4.9890e-02, -7.4511e-02, -1.7695e-01],
           [-5.2205e-02, -1.2139e-01, -1.1745e-01]],

          [[ 1.4240e-01, -7.7569e-02, -6.2362e-02],
           [ 1.3522e-01, -2.5257e-02, -8.7782e-02],
           [-1.3324e-01,  9.0810e-03, -1.0441e-01]],

          [[ 1.4080e-01,  1.3825e-01,  1.2576e-01],
           [ 2.0621e-01,  1.2434e-01, -2.9911e-03],
           [ 5.9175e-02, -4.5880e-02,  9.5690e-02]]],


 

In [118]:
## create a funciton of the above code

from nets.unet import Unet


def add_input_to_pre_trained(model: dict) -> dict:
    """Add additional input channel to the first layer of a pre trained model"""

    # input size of first layer
    input_size = model["conv_1.conv.unit0.conv.weight"].shape[1]
    print(
        f'Input channels before update: {model["conv_1.conv.unit0.conv.weight"][1][3]}'
    )
    print(f"Input channels of model: {input_size}")
    # additinal input channel
    num_input_channels = input_size + 1
    # create a new Conv3d layer with an additional input channel
    new_conv = torch.nn.Conv3d(
        num_input_channels, 16, kernel_size=3, stride=1, padding=1, bias=True
    )
    # copy the weights from the old conv layer to the new conv layer
    with torch.no_grad():
        new_conv.weight[:, :input_size, :, :, :] = model[
            "conv_1.conv.unit0.conv.weight"
        ]
        new_conv.bias = torch.nn.Parameter(
            model["conv_1.conv.unit0.conv.bias"]
        )  # Convert to torch.nn.Parameter
    # replace the old conv layer with the new conv layer in the model
    model["conv_1.conv.unit0.conv.weight"] = new_conv.weight
    model["conv_1.conv.unit0.conv.bias"] = new_conv.bias

    print(
        f'Input channels of model after update: {model["conv_1.conv.unit0.conv.weight"].shape[1]}'
    )
    print(
        f'Input channels of model after update: {model["conv_1.conv.unit0.conv.weight"][1][4]}'
    )

    return model


print("Running on GPU:" + str(1))
print("Running for epochs:" + str(10))
cuda_id = "cuda:" + str(1)
device = torch.device(cuda_id)
torch.cuda.set_device(cuda_id)


pre_trained_model = (
    "models/all_in_one/TBI/standard_random_drop_1_2024-12-17_11-07_Epoch_599.pth"
)
checkpoint = torch.load(pre_trained_model)

device = 0
model = Unet(in_channels=3).to(device)


model_update = add_input_to_pre_trained(model)

model_update.load_state_dict(checkpoint["model_state_dict"])

Running on GPU:1
Running for epochs:10
RES_UNET INIT
Dropout:  0.2


TypeError: 'Unet' object is not subscriptable

In [ ]:
## looking through dictionary and swwapping a value of a specific key 

def swap_value_in_dict(dictionary: dict, key: str, value: str) -> dict:
    """Swap the value of a specific key in a dictionary"""
    
    dictionary[key] = value
    
    return

In [ ]:
### Adversarial loss for one channel of a modelity 

# is it possible to have simple linear neural network for one channel of a model and then fuse the output with various other channels of the modality 
# Augment data from what data distribution.
# pixel contrast and resolution want to augment to a sensible distribution. 
# want to augment noise (pathologies) 




In [ ]:
# data augmentation for invariant slot. 
# 
